In [1]:
from src.log_config import configure_logging
from src.generate_data import generate_mock_data, MOCK_DATA_FOLDER, DATA_NAMES
from src.io import load_json_to_dataframe, save_csv
from src.transform import auto_correct_regions
from src.validate import validate_temperature_records

#Configure logger
configure_logging()

#Generate 5 mock data with different problems.
generate_mock_data()


Saved 5 mock data json files to data/


# Rapport: Valideringspipeline för Sensordata
**Författare:** Marcus Bäckström  
**Kurs:** Valfri fördjupning inom Python för Data Science  

## 1. Syfte och Relevans
Syftet med projektet är att bygga en automatiserad pipeline som kvalitetssäkrar inkommande sensordata innan den hamnar i en databas eller används för ML-modeller. 

Inom Data Science är datakvalitet avgörande ("Garbage in, garbage out"). Om oren data släpps igenom havererar analyser och modeller. Pipelinen hanterar detta genom automatiska korrigeringar samt isolering av felaktiga rader för manuell granskning (Human-in-the-Loop).

## 2. Centrala Begrepp & Teori
* **Pydantic:** Används för strikt schemavalidering av datatyper och gränsvärden på radnivå.
* **Pandas:** Används för vektoriserad och effektiv datarensning (auto-correct) innan validering.
* **Deterministisk städning vs Validering:** Kända, entydiga fel (som felstavade regioner) rättas automatiskt i Pandas. Okända fel eller orimliga temperaturer skickas till Pydantic där de underkänns.
* **Human-in-the-Loop:** Istället för att krascha programmet när fel uppstår, flaggas raderna (`flagged_for_manual_review = True`) för manuell hantering.

## 3. Genomförande & Demonstration
Nedan demonstreras pipelinen steg för steg på datasetet `mixed_failure_data.json`. Datasetet innehåller avsiktligt flera typer av fel: felstavade regioner, felaktiga sektorer och orimliga temperaturvärden.

### Steg 1: Inläsning av Rådata

In [2]:
df_raw = load_json_to_dataframe(f"{MOCK_DATA_FOLDER}/mixed_failure_data.json")
df_raw

2026-09-18 13:18:39 | INFO | temperature_log | Successfully read JSON file at: data\mixed_failure_data.json


,sensor_id,region,sector,temperature
0,S1,Örebro,West,20.0
1,S2,Orebro,C,51.2
2,S3,orobo,SE,20.3
3,S4,Orebro,Söder,30.5
4,S5,sthlm,C,10.0


### Steg 2: Deterministisk Städning (Pandas)
Kända och entydiga fel (som teckenkodningsfel i regionnamn) korrigeras vektoriserat i Pandas via en dict-mappning. Kolumnen `original_region` sparas för spårbarhet.

In [3]:
df_cleaned = auto_correct_regions(df_raw)
df_cleaned

2026-09-18 13:18:47 | INFO | temperature_log | Auto-corrected 3 region entries.


,sensor_id,region,sector,temperature,original_region,auto_cleaned
0,S1,Örebro,West,20.0,None,False
1,S2,Örebro,C,51.2,Orebro,True
2,S3,orobo,SE,20.3,None,False
3,S4,Örebro,Söder,30.5,Orebro,True
4,S5,Stockholm,C,10.0,sthlm,True


### Steg 3: Schemavalidering & Flaggning (Pydantic)
Datan skickas genom Pydantic-modellen `TemperatureRead`. Rader som bryter mot datatyper eller gränsvärden (-40°C till +40°C) kraschar inte pipelinen, utan isoleras med `flagged_for_manual_review = True`.

In [4]:
df_validated = validate_temperature_records(df_cleaned, "mixed_failure_data")
df_validated

2026-09-18 13:18:58 | WARNING | temperature_log | in mixed_failure_data: 3 of 5 rows were flagged for manual review.


,sensor_id,region,sector,temperature,original_region,auto_cleaned,flagged_for_manual_review
0,S1,Örebro,West,20.0,None,False,False
1,S2,Örebro,C,51.2,Orebro,True,True
2,S3,orobo,SE,20.3,None,False,True
3,S4,Örebro,Söder,30.5,Orebro,True,True
4,S5,Stockholm,C,10.0,sthlm,True,False


## 4. Resultat och Sammanfattning av körning
Pipelinen körs på samma sätt över alla 5 genererade mock-dataset. Resultatet blir att giltiga rader bevaras, kända fel auto-korrigeras och okända/orimliga avvikelser isoleras utan avbrott i exekveringen.

In [ ]:
for data_name in DATA_NAMES:

    #Load Json files into Pandas DataFrame.
    df_mock = load_json_to_dataframe(file_path=f"{MOCK_DATA_FOLDER}/{data_name}.json")

    #Correct any common known misspelling by dict variable mapping
    auto_corrected_df_mock = auto_correct_regions(df_mock)

    #Validate dataframe against pydantic schema in schemas.py
    validated_df_mock = validate_temperature_records(auto_corrected_df_mock, data_name)

    #Save dataframes into folder output as CSV
    save_csv(file_name=data_name,df=validated_df_mock)    